# molprop-ensemble — Notebook STABIL (sekuensial, per unit kerja)

Versi **sederhana & tahan-macet** dari `main.ipynb`. Prinsip inti:

- **Satu proses OS terpisah untuk SETIAP (model × dataset × seed).** Bukan satu proses
  panjang yang mengulang puluhan kombinasi di dalam dirinya. Tiap seed dapat CUDA
  context & memori GPU yang **bersih** (dijamin OS saat proses keluar) — ini menghapus
  penyebab hang "sukses 25 kombinasi lalu macet total" yang terjadi di notebook lama.
- **TIDAK ada paralel/sibling.** Semua sekuensial, satu per satu. Lebih lambat sedikit,
  tapi kalau macet Anda tahu PERSIS kombinasi mana, dan proses lain tak ikut mati.
- **Resume otomatis.** Kombinasi yang prediksinya sudah ada di-`skip`. Kalau satu seed
  macet/timeout, ia **dilewati** (bukan menghentikan yang lain); jalankan ULANG sel itu —
  yang sudah selesai di-skip, yang gagal di-retry.
- **Versi library DIPIN** ke kombinasi yang terbukti sukses (`environment.txt` run asli):
  transformers 5.0.0, tokenizers 0.22.2, rdkit 2026.3.3, chemprop 2.2.4.
- **Hasil di-push ke GitHub BERTAHAP** (setelah tiap milestone), bukan cuma di paling akhir —
  jadi hasil yang sudah pasti tak hilang walau tahap berikutnya bermasalah.

**Prasyarat panel kanan:** Accelerator = **GPU T4 x2**, Internet = **On**, Secret **`GH_TOKEN`**.
Cukup **Run All**. Kalau sesi putus, Run All lagi — lanjut dari titik terakhir.

## 1. Setup — clone, install (versi DIPIN), cek environment

In [ ]:
# 1a. Clone / pull repo
REPO_OWNER, REPO_NAME, REPO_DIR = "belvahector-ship-it", "pharm_", "pharm_"
import os, subprocess
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    url = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if token
           else f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git")
    print("clone via GH_TOKEN..." if token else "GH_TOKEN tak ada -> clone publik.")
    r = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True,
                       timeout=120, stdin=subprocess.DEVNULL)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr); raise RuntimeError("Clone gagal — cek GH_TOKEN / KAGGLE.md.")
else:
    print(f"{REPO_DIR} sudah ada, skip clone.")
%cd {REPO_DIR}
!git pull --quiet && git log --oneline -1

In [ ]:
# 1b. Install dependency — versi DIPIN ke yang terbukti sukses (environment.txt).
# Torch TIDAK di-reinstall (pakai bawaan Kaggle 2.10.0+cu128 yg sudah terbukti jalan).
!pip install -q "rdkit==2026.3.3" "chemprop==2.2.4" "transformers==5.0.0" "tokenizers==0.22.2"
print("install selesai (versi dipin, bukan 'latest' PyPI)")

In [ ]:
# 1c. Cek environment
import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  GPU", i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, "Set Accelerator = GPU T4 x2 di panel kanan, lalu restart."
import rdkit, sklearn, transformers, chemprop
print("rdkit", rdkit.__version__, "| sklearn", sklearn.__version__,
      "| transformers", transformers.__version__, "| chemprop", chemprop.__version__)

## 2. Data & scaffold split

In [ ]:
!python scripts/01_prepare_data.py
!python scripts/00_diagnose_split.py

## 3. Runner sekuensial — inti notebook ini

`run_jobs()` menjalankan sebuah script SEKALI PER (dataset, seed) sebagai **proses OS
terpisah**, satu per satu. Tiap proses fresh -> tak ada akumulasi state -> tak ada hang
"menumpuk". `timeout_sec` = jaring pengaman: kalau satu kombinasi menggantung, ia di-kill
& dilewati (bukan mengunci seluruh sel selamanya).

In [ ]:
import subprocess, time

def run_jobs(script, jobs, model=None, timeout_sec=1800):
    """Jalankan `script --datasets D --seeds S` (plus `--model M` bila diberi) SEKALI PER
    (dataset, seed) di `jobs`. Proses OS TERPISAH & SEKUENSIAL tiap kombinasi (BUKAN paralel).
    Gagal/timeout TIDAK menghentikan kombinasi lain -> dicatat & dilewati (resume otomatis
    retry saat sel dijalankan ulang). Return daftar (tag, alasan) yang gagal."""
    fails = []
    for i, (dataset, seed) in enumerate(jobs, 1):
        args = ["python", script, "--datasets", dataset, "--seeds", str(seed)]
        if model:
            args[2:2] = ["--model", model]
        tag = f"{model or script.split('/')[-1]} | {dataset} | seed {seed}"
        print(f"\n[{i}/{len(jobs)}] === {tag} ===", flush=True)
        t0 = time.time()
        try:
            rc = subprocess.run(args, timeout=timeout_sec).returncode
            dt = time.time() - t0
            print(f"  {'OK' if rc==0 else 'GAGAL rc='+str(rc)} ({dt:.0f}s)", flush=True)
            if rc != 0:
                fails.append((tag, f"rc={rc}"))
        except subprocess.TimeoutExpired:
            fails.append((tag, f"timeout>{timeout_sec}s"))
            print(f"  TIMEOUT >{timeout_sec}s -> proses di-kill, dilewati", flush=True)
    return fails

def lapor(nama, fails):
    if fails:
        print(f"\n!! {nama}: {len(fails)} kombinasi BELUM selesai (dilewati):")
        for tag, why in fails:
            print(f"   - {tag}: {why}")
        print("   -> jalankan ULANG sel ini: resume skip yang sudah selesai, retry yang ini saja.")
    else:
        print(f"\n{nama}: SEMUA kombinasi selesai.")

def run_once(script, timeout_sec=1800):
    """Jalankan script SEKALI (untuk tahap post-processing ringan: fusion/evaluate/tuning)."""
    print(f">>> {script}", flush=True)
    rc = subprocess.run(["python", script], timeout=timeout_sec).returncode
    assert rc == 0, f"{script} gagal (rc={rc})"

import config
JOBS = [(d, s) for d in config.DATASETS for s in config.SEEDS]
print("Datasets:", config.DATASETS, "| Seeds:", config.SEEDS)
print("Total kombinasi per model:", len(JOBS))

## 4. Training baseline — SATU model, SATU dataset, SATU seed per proses

Tiga sel di bawah (ChemBERTa / D-MPNN / RF) masing-masing melatih SEMUA (dataset × seed)
satu-per-satu. Jalankan berurutan. Kalau salah satu macet di tengah, jalankan ulang sel
itu saja — yang sudah selesai otomatis di-skip.

In [ ]:
# 4a. ChemBERTa (GPU 0)
fails = run_jobs("scripts/02_train_baselines.py", JOBS, model="chemberta")
lapor("ChemBERTa", fails)

In [ ]:
# 4b. D-MPNN (GPU 1) — chemprop punya timeout internal 900s per training (jaring pengaman ganda)
fails = run_jobs("scripts/02_train_baselines.py", JOBS, model="dmpnn")
lapor("D-MPNN", fails)

In [ ]:
# 4c. ECFP + RF (CPU, cepat)
fails = run_jobs("scripts/02_train_baselines.py", JOBS, model="rf")
lapor("RF", fails)

## 5. TTA (ChemBERTa) — juga per (dataset, seed), satu proses masing-masing

In [ ]:
fails = run_jobs("scripts/04_run_tta.py", JOBS)   # script 04 tak butuh --model
lapor("TTA", fails)

## 6. Fusion + Evaluasi (post-processing, cepat, sekali jalan) -> tabel hasil tes1

In [ ]:
run_once("scripts/03_run_fusion.py")
run_once("scripts/05_evaluate.py")

import pandas as pd
pd.set_option("display.max_rows", None, "display.width", 200)
print("\n=== Tabel hasil TES 1 ===")
display(pd.read_csv("outputs/results/final_table.csv"))

## 7. Simpan hasil tes1 ke GitHub (permanen) — SEBELUM tahap lanjutan

Push bertahap: hasil baseline yang sudah pasti langsung diarsipkan, tak menunggu seluruh
notebook selesai. Kalau tahap di bawah bermasalah, hasil ini sudah aman.

In [ ]:
from src.utils.git_push import push_results
push_results("tes1", REPO_OWNER, REPO_NAME)

## 8. Tuning v1 & v2 + Analisis post-hoc (Kategori A) — semua CPU, tanpa retraining

Murni post-processing dari prediksi yang sudah ada. Cepat & aman (tak ada risiko hang GPU).

In [ ]:
run_once("scripts/07_evaluate_tuned.py")     # tuning v1: adaptive TTA gating
run_once("scripts/08_evaluate_best.py")      # tuning v2: subset selektif + kalibrasi
run_once("scripts/09_posthoc_analysis.py")   # Kategori A: PR-AUC, Holm-Bonferroni, dll

import pandas as pd
print("\n=== Tuned v2 (terbaik per dataset) ===")
display(pd.read_csv("outputs/results/tuned_v2_best/final_table_best.csv"))
print("=== Instance-level TTA gate (proxy) ===")
display(pd.read_csv("outputs/results/posthoc/instance_level_tta_gate_summary.csv"))

In [ ]:
from src.utils.git_push import push_results
push_results("tuned+posthoc", REPO_OWNER, REPO_NAME)

## 9. (OPSIONAL) Category C — Improvement v3 (retraining: Focal Loss + EMA + binary-mcc)

Bagian ini **melatih ulang** (chemberta_v3 / dmpnn_v3), jadi paling berat & paling rawan.
Hasil di atas SUDAH ter-push permanen, jadi bagian ini murni bonus — aman kalau mau di-skip.
Tetap per (dataset × seed) satu proses, sekuensial, resume-safe.

In [ ]:
# 9a. Training v3 (chemberta_v3 GPU0, dmpnn_v3 GPU1) — sekuensial, per seed
fails = run_jobs("scripts/10_train_v3.py", JOBS, model="chemberta_v3")
lapor("chemberta_v3", fails)
fails = run_jobs("scripts/10_train_v3.py", JOBS, model="dmpnn_v3")
lapor("dmpnn_v3", fails)

In [ ]:
# 9b. Instance-level TTA gate PENUH (dua backbone) — per seed
fails = run_jobs("scripts/11_run_tta_v3.py", JOBS)
lapor("TTA v3 (instance-gate)", fails)

In [ ]:
# 9c. Ensemble v3 + tabel akhir (post-processing, sekali jalan)
run_once("scripts/12_fuse_evaluate_v3.py")

import pandas as pd
print("\n=== Tabel hasil v3 ===")
display(pd.read_csv("outputs/results/v3/final_table_v3.csv"))

In [ ]:
from src.utils.git_push import push_results
push_results("category-c-v3", REPO_OWNER, REPO_NAME)

## 10. Cadangan — zip semua output & link unduh

In [ ]:
import os, shutil
from IPython.display import FileLink, display
!pip freeze > outputs/results/pip_freeze.txt
zip_name = "hasil_outputs"
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")
shutil.make_archive(zip_name, "zip", "outputs")
mb = os.path.getsize(f"{zip_name}.zip") / (1024*1024)
print(f"Zip: {zip_name}.zip ({mb:.1f} MB) — klik link di bawah, atau panel kanan -> Output.")
display(FileLink(f"{zip_name}.zip"))